# scRNA-seq 분석 실습

건국대학교 이형우 교수님 연구실 온라인 세미나
Data: **GSE210543** — Human Retina (Young vs Old)
Runtime: **Google Colab + Seurat v5 (R)**

## 0. 데이터셋 개요 — GSE210543

### Cell Ranger Web Summary

10X Genomics Cell Ranger가 시퀀싱 완료 후 생성하는 **QC 리포트**입니다.
분석 시작 전 반드시 확인해야 하는 **3가지 지표**:

| 지표 | 의미 | 권장 기준 |
|------|------|---------|
| **Estimated Number of Cells** | Cell Ranger 추정 세포 수 | **> 2,000** |
| **Median Genes per Cell** | 세포당 검출 유전자 중앙값 | **> 1,500** |
| **Fraction Reads in Cells** | 세포에 할당된 reads 비율 | **> 70%** |

> **💡 Tip:** Mean Reads per Cell이 매우 높다면 세포 수가 너무 적어서 생기는 **수학적 효과**일 수 있습니다 — 단독으로 해석하지 마세요.

### 전체 13개 샘플 품질 요약

| 샘플 | 그룹 | 세포 수 | Mean Reads/Cell | Median Genes/Cell | 선택 |
|------|------|-------:|---------------:|------------------:|:----:|
| **16PCW** | Young | **8,601** | 43,243 | 1,084 | ✅ |
| **20PCW** | Young | **5,787** | 103,036 | **5,174** | ✅ |
| 12PCW | Young | 3,637 | 164,166 | 4,596 | — |
| 21PCW | Young | 3,948 | 139,139 | 1,953 | — |
| **Adult_2** | Old | **6,516** | 90,897 | 2,016 | ✅ |
| **Adult_3** | Old | **3,694** | 142,303 | 2,806 | ✅ |
| Adult_5 | Old | 3,487 | 168,631 | 2,529 | — |
| Adult_1 | Old | 884 | 412,206 | 2,807 | ❌ |
| Adult_4 | Old | 496 | 1,146,570 | 2,414 | ❌ |
| AMD_macula | AMD | 1,719 | 284,749 | 4,002 | — |
| AMD_peripheral | AMD | 1,794 | 267,813 | 3,584 | — |
| Unaffected_macula | Control | 3,557 | 101,143 | 5,102 | — |
| Unaffected_peripheral | Control | 3,527 | 112,731 | 4,765 | — |

✅ 세미나 분석 샘플 (Young 2 + Old 2) | ❌ 세포 수 부족 또는 품질 이슈

### Web Summary 읽는 법 — 좋은 샘플 vs 나쁜 샘플

**✅ 좋은 샘플 — 20PCW**

<table width="100%" cellpadding="0" cellspacing="0" style="background: #F0FDF4; border: 2px solid #16A34A; border-radius: 10px; margin: 18px 0 4px 0;">
<tr>
  <td style="padding: 14px 20px; vertical-align: middle;">
    <span style="background: #16A34A; color: white; padding: 3px 12px; border-radius: 12px; font-size: 12px; font-weight: bold; margin-right: 10px;">GOOD</span>
    <span style="font-size: 17px; font-weight: 700; color: #14532D;">20PCW — 이상적인 라이브러리</span>
  </td>
  <td align="right" style="padding: 14px 20px; color: #166534; font-size: 13px; white-space: nowrap; vertical-align: middle;">5,787 cells &middot; 5,174 genes/cell &middot; 91.3% in cells</td>
</tr>
</table>

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/websummary_good_20PCW.png" width="900"/>

| 항목 | 값 | 평가 |
|------|-----|------|
| Estimated Cells | **5,787** | ✅ 충분 |
| Mean Reads/Cell | 103,036 | ✅ 정상 범위 |
| Median Genes/Cell | **5,174** | ✅ 매우 우수 |
| Fraction in Cells | **91.3%** | ✅ 배경 노이즈 최소 |

Barcode Rank Plot에서 파란 선과 회색 선 사이의 **꺾임(knee)**이 뚜렷합니다.

---

**❌ 나쁜 샘플 — Adult_4**

<table width="100%" cellpadding="0" cellspacing="0" style="background: #FFF1F2; border: 2px solid #E11D48; border-radius: 10px; margin: 18px 0 4px 0;">
<tr>
  <td style="padding: 14px 20px; vertical-align: middle;">
    <span style="background: #E11D48; color: white; padding: 3px 12px; border-radius: 12px; font-size: 12px; font-weight: bold; margin-right: 10px;">BAD</span>
    <span style="font-size: 17px; font-weight: 700; color: #881337;">Adult_4 — 세포 포획 실패</span>
  </td>
  <td align="right" style="padding: 14px 20px; color: #BE123C; font-size: 13px; white-space: nowrap; vertical-align: middle;">496 cells &middot; 1,146,570 reads/cell</td>
</tr>
</table>

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/websummary_bad_Adult4.png" width="900"/>

| 항목 | 값 | 평가 |
|------|-----|------|
| Estimated Cells | **496** | ❌ 세포 포획 실패 |
| Mean Reads/Cell | **1,146,570** | ⚠️ 비정상 (수학적 효과) |
| Median Genes/Cell | 2,414 | ⚠️ 보통 |
| Fraction in Cells | 78.8% | ⚠️ 낮은 편 |

```
Mean Reads/Cell = 총 reads ÷ 세포 수
               = 568,698,782 ÷ 496 ≈ 1,146,570
```

→ 총 reads는 20PCW(596M)와 비슷하지만 세포 수가 극히 적어 값이 폭등.
**세포 포획 실패**가 원인 — 분석에서 제외.

## Step 0. 환경 설정

> **Colab 설정:** Runtime → Change runtime type → **R**

패키지 설치는 처음 1번만 실행합니다 (~10분 소요).

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_04.png" width="850"/>

*Fig. 0 — scRNA-seq 분석의 복잡성 (Hicks et al., BioRxiv 2015)*

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_05.png" width="850"/>

*Fig. 1 — scRNA-seq 전체 분석 워크플로우 (Luecken & Theis, Mol Syst Biol 2019)*

In [ ]:
# ╔═══════════════════════════════════════════════════════╗
# ║  강사 전용 — 세미나 전 1회 실행                          ║
# ║  참가자는 이 셀을 건너뛰고 다음 셀을 실행하세요!          ║
# ╚═══════════════════════════════════════════════════════╝

# Drive 마운트
system("python3 -c \"from google.colab import drive; drive.mount('/content/drive')\"")

# 공유 폴더에 패키지 설치 (약 15~20분, 1회만)
lib_path <- "/content/drive/MyDrive/KU_seminar/R_libs"
dir.create(lib_path, recursive = TRUE, showWarnings = FALSE)

install.packages(
  c("Seurat", "harmony", "ggplot2", "patchwork"),
  lib   = lib_path,
  repos = "https://cloud.r-project.org"
)

cat("\n설치 완료! 참가자들이 Drive에서 바로 로드할 수 있습니다.\n")
cat("경로:", lib_path, "\n")


In [ ]:
# ▶ Step 1. Google Drive 마운트
# 실행 후 팝업이 뜨면 Google 계정 선택 → '허용' 클릭
# '마운트됨' 메시지가 나오면 다음 셀을 실행하세요!

system("python3 -c \"from google.colab import drive; drive.mount('/content/drive')\"")


In [ ]:
# ▶ Step 2. Drive 패키지 로드 (설치 불필요!)
# 위 셀에서 '마운트됨' 확인 후 실행하세요

lib_path <- "/content/drive/MyDrive/KU_seminar/R_libs"
.libPaths(c(lib_path, .libPaths()))

library(Seurat)
library(harmony)
library(ggplot2)
library(patchwork)

set.seed(42)
cat("준비 완료!\n")
cat("Seurat:", as.character(packageVersion("Seurat")), "\n")


<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_06.png" width="850"/>

*Fig. 2 — Seurat 기반 분석 워크플로우*

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_07.png" width="850"/>

*Fig. 3 — Seurat vs Scanpy 비교 및 Tip 1*

## Step 1. 데이터 불러오기

10X Genomics `filtered_feature_bc_matrix` 폴더를 읽어 **Seurat 오브젝트**를 생성합니다.

```
filtered_feature_bc_matrix/
├── barcodes.tsv.gz   ← 세포 바코드
├── features.tsv.gz   ← 유전자 ID / 이름
└── matrix.mtx.gz     ← UMI count matrix (sparse)
```

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_08.png" width="850"/>

*Fig. 4 — Count Matrix 생성 원리: barcodes / features / matrix (Macosko et al., Cell 2015)*

In [ ]:
# ── Google Drive 마운트 ───────────────────────────────────────────
# Colab 좌측 파일 아이콘(📁) → 드라이브 아이콘 클릭해 마운트 후 아래 실행
# 또는 아래 코드로 자동 마운트 (브라우저 인증 팝업 발생)

system("python3 -c \"from google.colab import drive; drive.mount('/content/drive')\"")

# ──────────────────────────────────────────────────────────────────
# ▶ 참가자 안내: 세미나 데이터 접근 방법
#
#   1. 공유 링크 열기:
#      https://drive.google.com/drive/folders/1CM6A8GEMVIfpmwZTyHlzaudDCRYchYII
#
#   2. 폴더 우클릭 → "내 드라이브에 바로가기 추가"
#      → 내 드라이브 최상위에 저장 → 이름: KU_seminar
#
#   3. 아래 GDRIVE_FOLDER 경로가 자동으로 맞습니다
# ──────────────────────────────────────────────────────────────────

GDRIVE_FOLDER <- "/content/drive/MyDrive/KU_seminar"

# 압축 해제 (처음 1회만 — 이미 해제되어 있으면 스킵)
DATA_DIR <- "/content/02.filtered_feature_bc_matrix"

if (!dir.exists(DATA_DIR)) {
  zip_path <- file.path(GDRIVE_FOLDER, "02.filtered_feature_bc_matrix.zip")
  cat("압축 해제 중 (약 2~3분 소요)...\n")
  ret <- system(paste0("unzip -q '", zip_path, "' -d /content/"))
  if (ret == 0) {
    cat("완료!\n")
  } else {
    stop("압축 해제 실패 — GDRIVE_FOLDER 경로를 확인하세요: ", GDRIVE_FOLDER)
  }
} else {
  cat("이미 압축 해제됨 — 스킵\n")
}

cat("\n데이터 경로:", DATA_DIR, "\n")
cat("샘플 목록:\n")
samples_found <- list.dirs(DATA_DIR, recursive = FALSE, full.names = FALSE)
cat(paste(" ", samples_found, collapse = "\n"), "\n")

In [ ]:
# 4개 샘플을 하나씩 로드합니다
# Read10X: 10X Genomics 출력 파일(barcodes, features, matrix)을 읽어옵니다

# ── Young 샘플 ──────────────────────────────────────────────────
counts <- Read10X(data.dir = file.path(DATA_DIR, "16PCW"))
s_Young_16PCW <- CreateSeuratObject(counts = counts, project = "Young_16PCW",
                                    min.cells = 3, min.features = 200)
s_Young_16PCW$sample <- "Young_16PCW"
s_Young_16PCW$group  <- "Young"

counts <- Read10X(data.dir = file.path(DATA_DIR, "20PCW"))
s_Young_20PCW <- CreateSeuratObject(counts = counts, project = "Young_20PCW",
                                    min.cells = 3, min.features = 200)
s_Young_20PCW$sample <- "Young_20PCW"
s_Young_20PCW$group  <- "Young"

# ── Old 샘플 ────────────────────────────────────────────────────
counts <- Read10X(data.dir = file.path(DATA_DIR, "Adult_2"))
s_Old_Adult2 <- CreateSeuratObject(counts = counts, project = "Old_Adult2",
                                   min.cells = 3, min.features = 200)
s_Old_Adult2$sample <- "Old_Adult2"
s_Old_Adult2$group  <- "Old"

counts <- Read10X(data.dir = file.path(DATA_DIR, "Adult_3"))
s_Old_Adult3 <- CreateSeuratObject(counts = counts, project = "Old_Adult3",
                                   min.cells = 3, min.features = 200)
s_Old_Adult3$sample <- "Old_Adult3"
s_Old_Adult3$group  <- "Old"

# 목록으로 묶기
seurat_list <- list(
  Young_16PCW = s_Young_16PCW,
  Young_20PCW = s_Young_20PCW,
  Old_Adult2  = s_Old_Adult2,
  Old_Adult3  = s_Old_Adult3
)

# 결과 확인
cat("샘플별 세포 수:\n")
cat("  Young_16PCW:", ncol(s_Young_16PCW), "cells\n")
cat("  Young_20PCW:", ncol(s_Young_20PCW), "cells\n")
cat("  Old_Adult2 :", ncol(s_Old_Adult2),  "cells\n")
cat("  Old_Adult3 :", ncol(s_Old_Adult3),  "cells\n")


**▶ 결과 해석**

각 샘플의 세포 수를 확인하세요:

- `CreateSeuratObject(min.cells=3)` — 최소 3개 세포에서 발현된 유전자만 포함
- `min.features=200` — 200개 미만 유전자 발현 세포는 이 단계에서 제거됨
- Cell Ranger 추정치보다 낮을 수 있음 (엄격한 초기 필터링 효과)

> **Young 샘플** (16PCW, 20PCW): 발달기 세포, 증식 세포 혼재 가능
> **Old 샘플** (Adult_2, Adult_3): 성체 성숙 세포, 구성이 안정적

## Step 2. QC — Quality Control

### 핵심 QC 지표 3가지

| 지표 | 의미 | 기본 필터 |
|------|------|---------|
| `nFeature_RNA` | 세포당 검출 유전자 수 | **> 200** |
| `nCount_RNA` | 세포당 총 UMI 수 | **> 500** |
| `percent.mt` | 미토콘드리아 유전자 비율 | **< 10%** |

**낮은 nFeature** → 빈 droplet 또는 죽은 세포
**높은 percent.mt** → 세포막 손상, 세포질 RNA 유출

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_09.png" width="850"/>

*Fig. 5 — QC 지표 분포 확인 방법 (NCells, nUMI, nGene, mitoRatio)*

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_10.png" width="850"/>

*Fig. 6 — QC VlnPlot 예시: nFeature_RNA / nCount_RNA / percent.mt*

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_11.png" width="850"/>

*Fig. 7 — QC 필터링 실습 코드 및 Tip 2: Filter 조건에서의 정답은 없다!*

In [ ]:
# 미토콘드리아 유전자 비율 계산
# 인간 미토콘드리아 유전자는 이름이 'MT-' 로 시작합니다
# 높은 비율 = 세포막이 손상되어 세포질 mRNA가 빠져나간 상태 (죽은 세포)

seurat_list[["Young_16PCW"]][["percent.mt"]] <- PercentageFeatureSet(seurat_list[["Young_16PCW"]], pattern = "^MT-")
seurat_list[["Young_20PCW"]][["percent.mt"]] <- PercentageFeatureSet(seurat_list[["Young_20PCW"]], pattern = "^MT-")
seurat_list[["Old_Adult2"]][["percent.mt"]]  <- PercentageFeatureSet(seurat_list[["Old_Adult2"]],  pattern = "^MT-")
seurat_list[["Old_Adult3"]][["percent.mt"]]  <- PercentageFeatureSet(seurat_list[["Old_Adult3"]],  pattern = "^MT-")

# QC 지표 확인 (첫 6개 세포 예시)
head(seurat_list[["Young_16PCW"]]@meta.data[, c("nFeature_RNA", "nCount_RNA", "percent.mt")])


In [ ]:
# QC 분포 시각화
# 4개 샘플을 임시로 합쳐서 한 번에 비교합니다

seurat_qc_merged <- merge(
  seurat_list[["Young_16PCW"]],
  y = list(seurat_list[["Young_20PCW"]],
           seurat_list[["Old_Adult2"]],
           seurat_list[["Old_Adult3"]]),
  add.cell.ids = names(seurat_list)
)
seurat_qc_merged[["percent.mt"]] <- PercentageFeatureSet(seurat_qc_merged, pattern = "^MT-")

VlnPlot(
  seurat_qc_merged,
  features = c("nFeature_RNA", "nCount_RNA", "percent.mt"),
  group.by = "sample",
  ncol     = 3,
  pt.size  = 0
) & theme(axis.text.x = element_text(angle = 45, hjust = 1))


In [ ]:
# QC 통계 요약 — 샘플별로 확인합니다
# 이 숫자를 보고 아래 필터링 임계값을 결정하세요

for (name in names(seurat_list)) {
  obj <- seurat_list[[name]]
  cat("──", name, "──────────────────────\n")
  cat("  세포 수         :", ncol(obj), "\n")
  cat("  nFeature 중앙값 :", round(median(obj$nFeature_RNA)), "\n")
  cat("  nFeature 95%ile :", round(quantile(obj$nFeature_RNA, 0.95)), "\n")
  cat("  nCount 중앙값   :", round(median(obj$nCount_RNA)), "\n")
  cat("  nCount 98%ile   :", round(quantile(obj$nCount_RNA, 0.98)), " ← Singlet 기준\n")
  cat("  MT% 중앙값      :", round(median(obj$percent.mt), 2), "\n")
  cat("\n")
}


**▶ 결과 해석**

VlnPlot과 통계에서 확인할 항목:

| 지표 | 낮은 값 문제 | 높은 값 문제 |
|------|------------|------------|
| `nFeature_RNA` | 빈 droplet / 죽은 세포 | **Doublet** (두 세포가 하나로 포획) |
| `nCount_RNA` | 낮은 품질 | Doublet 의심 |
| `percent.mt` | — | 세포막 손상 |

**nFeature 상한선 설정 기준**: 샘플의 **95th percentile** 또는 분포에서 뚜렷한 어깨(shoulder)가 보이는 지점

> **고급 QC — DoubletFinder**
> nFeature 상한선으로 doublet을 어느 정도 걸러낼 수 있지만, 더 정확한 방법은
> `DoubletFinder` 패키지를 사용하는 것입니다. 샘플별로 실행하며 각 세포에
> doublet 확률을 부여합니다. 3시간 세미나에서는 생략하지만, 실제 분석에서는
> QC 단계에 포함하는 것을 권장합니다.
> ```r
> # 참고 (실행 X)
> # remotes::install_github("chris-mcginnis-ucsf/DoubletFinder")
> # library(DoubletFinder)
> # pK <- 0.09  # 최적값은 paramSweep으로 결정
> # seurat_obj <- doubletFinder(seurat_obj, PCs=1:30, pN=0.25, pK=pK, nExp=...)
> ```

In [ ]:
# nFeature vs nCount 산점도 — doublet 탐지
# 정상 세포: 두 지표가 선형 관계
# Doublet: nCount, nFeature 모두 비정상적으로 높음

# Young_16PCW 예시
p1 <- FeatureScatter(seurat_list[["Young_16PCW"]],
                     feature1 = "nCount_RNA",
                     feature2 = "percent.mt") +
      ggtitle("Young_16PCW - Count vs MT%")

p2 <- FeatureScatter(seurat_list[["Young_16PCW"]],
                     feature1 = "nCount_RNA",
                     feature2 = "nFeature_RNA") +
      ggtitle("Young_16PCW - Count vs Feature")

p1 + p2


**▶ 결과 해석**

**Count vs MT%**: 정상 세포는 낮은 MT% 유지. MT% 가 높은 세포 → 세포막 손상 → 제거 대상
**Count vs Feature**: 정상 세포는 **선형 관계**를 보입니다.

- 오른쪽 상단 이상치 → UMI는 많은데 유전자 수도 많음 → **doublet 의심**
- 왼쪽 하단 이상치 → 유전자/UMI 모두 낮음 → **빈 droplet** 의심
- 선형 관계에서 벗어난 점들이 QC 필터링 대상입니다

## Step 3. QC 필터링

```r
nFeature_RNA > 200  &  nCount_RNA > 500  &  percent.mt < 10
```

> **💡 Tip:** 임계값은 샘플마다 다릅니다. VlnPlot을 보고 분포의 자연스러운 경계(valley)를 찾으세요.

In [ ]:
# QC 기준값 설정
# ▶ 위 통계를 확인하고 필요하면 수정하세요!
MIN_GENE <- 200   # 최소 유전자 수   (적으면 빈 droplet)
MIN_UMI  <- 500   # 최소 UMI 수
MAX_MT   <- 10    # 최대 미토콘드리아 비율 (%)
# UMI 상위 2%는 doublet(두 세포가 한 droplet)으로 간주 → 샘플별로 계산

seurat_list_filtered <- list()   # 필터링 결과를 저장할 빈 목록

for (name in names(seurat_list)) {
  obj <- seurat_list[[name]]

  # 이 샘플의 nCount 98th percentile (doublet 상한선)
  max_umi <- quantile(obj$nCount_RNA, 0.98)

  # 조건에 맞는 세포만 남기기
  obj_filtered <- subset(obj,
    subset = nFeature_RNA > MIN_GENE &
             nCount_RNA   > MIN_UMI  &
             nCount_RNA   < max_umi  &   # 상위 2% 제거
             percent.mt   < MAX_MT)

  seurat_list_filtered[[name]] <- obj_filtered

  cat(name, ":", ncol(obj), "→", ncol(obj_filtered), "cells",
      "(제거:", ncol(obj) - ncol(obj_filtered), ")\n")
}


**▶ 결과 해석**

**3단계 QC 결과** 확인:

| 제거 비율 | 해석 |
|---------|------|
| **5~15%** | 정상적인 QC |
| **> 25%** | 임계값 너무 엄격하거나 샘플 품질 이슈 |
| **< 3%** | 임계값 너무 느슨할 수 있음 |

**nCount 98th percentile (Singlet QC):**
- 상위 2% 세포는 **doublet 또는 triplet** (두 세포 이상이 한 droplet에 포획)
- 샘플별로 다른 값 적용 — 데이터 기반 필터링
- 고정값(5000 등)보다 **합리적이고 재현성 높음**

> **DoubletFinder** 는 더 정밀하지만 샘플별 10~20분 소요
> 실제 분석에서는 Singlet QC → DoubletFinder 순서로 적용 권장

## Step 4. 정규화 — Normalization

세포마다 **시퀀싱 깊이(sequencing depth)**가 다릅니다. 정규화로 이를 보정합니다.

**LogNormalize** (기본값):
```
normalized = log(count / total_count × 10,000 + 1)
```
→ 총 UMI 수로 나눈 뒤 **log 변환** — 깊이 차이를 제거하고 분포를 안정화

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_12.png" width="850"/>

*Fig. 8 — LogNormalization vs SCTransform 방법 비교 및 PC 선택 기준 (Tip 3)*

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_13.png" width="850"/>

*Fig. 9 — 정규화 개념: SCTransform vs LogNormalization (M. Loven, RNA-seq statistical analysis)*

In [ ]:
# 정규화: 각 세포의 총 UMI가 10,000이 되도록 보정한 후 log 변환
# → 세포마다 포획된 mRNA 양의 차이를 제거합니다

for (name in names(seurat_list_filtered)) {
  seurat_list_filtered[[name]] <- NormalizeData(
    seurat_list_filtered[[name]],
    normalization.method = "LogNormalize",
    scale.factor         = 10000
  )
}
cat("정규화 완료!\n")


## Step 5. HVG — 고변이 유전자 선택

전체 수만 개 유전자 중 **샘플 간 차이가 큰 유전자**만 선택하여 분석 효율을 높입니다.

일반적으로 **상위 2,000개** 사용 (Seurat 기본값).
→ 모든 유전자를 쓰면 노이즈가 커지고 속도가 느려집니다.

In [ ]:
# 고변동 유전자(HVG) 선택: 샘플 간 발현 차이가 큰 유전자 2,000개
# → 세포 타입 구분에 정보가 많은 유전자들입니다

for (name in names(seurat_list_filtered)) {
  seurat_list_filtered[[name]] <- FindVariableFeatures(
    seurat_list_filtered[[name]],
    selection.method = "vst",
    nfeatures        = 2000
  )
}

# Top 10 HVG 확인 (16PCW 예시)
top10 <- head(VariableFeatures(seurat_list_filtered[["Young_16PCW"]]), 10)
cat("Top 10 고변동 유전자 (Young_16PCW):\n")
print(top10)


In [ ]:
# QC + 정규화 + HVG가 완료된 4개 샘플을 하나로 합칩니다

seurat_merged <- merge(
  x            = seurat_list_filtered[["Young_16PCW"]],
  y            = list(seurat_list_filtered[["Young_20PCW"]],
                      seurat_list_filtered[["Old_Adult2"]],
                      seurat_list_filtered[["Old_Adult3"]]),
  add.cell.ids = names(seurat_list_filtered)
)

# Seurat v5 필수: 샘플별로 분리된 레이어를 하나로 합칩니다
seurat_merged <- JoinLayers(seurat_merged)

# 합친 뒤 전체 데이터에서 HVG 재선택
seurat_merged <- FindVariableFeatures(seurat_merged, nfeatures = 2000)

cat("합병 완료!\n")
cat("전체 세포 수  :", ncol(seurat_merged), "\n")
cat("전체 유전자 수:", nrow(seurat_merged), "\n")
cat("\n샘플별 세포 수:\n")
print(table(seurat_merged$sample))


**▶ 결과 해석**

`merge()` + `JoinLayers()` 후 4개 샘플이 하나의 오브젝트로 통합됩니다.

- **JoinLayers()**: Seurat v5에서 샘플별로 분리된 data 레이어를 하나로 합침
- **FindVariableFeatures()** 재실행: 병합 후 전체 세포에서 다시 HVG 선택
- 샘플별 세포 수 확인 → 크게 불균형하면 integration 품질에 영향

---

## 중간 체크포인트

QC + 정규화 + HVG까지 완료했습니다. 오브젝트를 저장합니다.

> **💡 Tip:** 세션이 끊기면 Step 0 라이브러리 로드 후 아래 **로드 셀**을 실행하세요.

In [ ]:
# 저장 경로 설정 (처음 1회 실행)
SAVE_DIR <- "/content/drive/MyDrive/KU_seminar/results"
dir.create(SAVE_DIR, showWarnings = FALSE, recursive = TRUE)

# 오브젝트 저장
saveRDS(seurat_merged, file = file.path(SAVE_DIR, "01_seurat_merged.rds"))
cat("저장 완료:", file.path(SAVE_DIR, "01_seurat_merged.rds"), "\n")

In [ ]:
# ── 세션 재시작 시 여기서부터 ────────────────────────────────────
# (위 셀들을 다시 실행하지 않아도 됩니다)

# library(Seurat); library(harmony); library(dplyr); library(ggplot2); library(patchwork)
# set.seed(42)
# SAVE_DIR <- "/content/drive/MyDrive/KU_seminar/results"
# seurat_merged <- readRDS(file.path(SAVE_DIR, "01_seurat_merged.rds"))
# cat("로드 완료:", ncol(seurat_merged), "cells\n")

## Step 6. Cell Cycle Scoring *(Optional)*

세포 주기(**G1 / S / G2M**)가 클러스터링에 영향을 줄 수 있습니다.
**발달기(Young)** 샘플에는 증식 세포가 많으므로 확인이 필요합니다.

→ 주기 효과가 클 경우 `vars.to.regress`로 회귀 제거 가능

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_14.png" width="850"/>

*Fig. 10 — Cell Cycle Scoring: 언제 회귀할지, 언제 남겨둘지*

In [ ]:
# Cell cycle 관련 유전자 (Seurat 내장)
s.genes   <- cc.genes$s.genes    # S phase
g2m.genes <- cc.genes$g2m.genes  # G2M phase

# Cell cycle score 계산
seurat_merged <- CellCycleScoring(
  seurat_merged,
  s.features   = s.genes,
  g2m.features = g2m.genes,
  set.ident    = TRUE
)

# 분포 확인
table(seurat_merged$Phase)

## Step 7. Scaling + PCA

**Scaling**: 유전자별 **평균 0, 분산 1**로 맞춰 발현량 크기 차이를 제거합니다.
**PCA**: 고변이 유전자 2,000개 → **주요 성분 50개**로 차원 축소.

→ PCA ElbowPlot으로 유효한 PC 수를 확인하세요 (보통 **15~30개** 사용)

In [ ]:
# Scaling: 유전자별 평균=0, 분산=1 정규화
# Cell cycle 효과 회귀 (발달기 샘플에 중요)
seurat_merged <- ScaleData(
  seurat_merged,
  vars.to.regress = c("S.Score", "G2M.Score"),
  features        = VariableFeatures(seurat_merged)  # HVG만 scale (속도 ↑)
)

cat("Scaling complete!\n")

In [ ]:
# PCA 실행
seurat_merged <- RunPCA(
  seurat_merged,
  features = VariableFeatures(seurat_merged),
  npcs     = 50
)

# Elbow Plot — 몇 개의 PC를 사용할지 결정
ElbowPlot(seurat_merged, ndims = 50) +
  ggtitle("Elbow Plot: PC 기여도") +
  geom_vline(xintercept = 30, linetype = "dashed", color = "red") +
  annotate("text", x = 32, y = 3, label = "PC=30 선택", color = "red")

**▶ 결과 해석**

**Elbow Plot**에서 표준편차가 급격히 감소하다가 완만해지는 **꺾임점(elbow)**을 찾으세요.

- 꺾임점 이후 PC들은 주로 **기술적 노이즈**를 반영합니다
- 이 데이터셋에서는 보통 **PC 20~30**이 적절합니다
- 의심스러우면 넉넉하게 (30) 선택 — 적게 쓰는 것보다 안전합니다

## Step 8. Integration — Harmony

**Young(발달기)**과 **Old(성체)** 샘플은 생물학적 차이 외에도 **배치 효과(batch effect)**가 있습니다.
**Harmony**로 배치를 보정하되 생물학적 신호는 보존합니다.

→ PCA space에서 샘플 레이블을 기준으로 반복 보정 수행

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_15.png" width="850"/>

*Fig. 11 — Integration 방법 비교: Harmony가 세포 타입 구조를 잘 보존 (Tran et al., Genome Biol 2020)*

### Harmony 통합 실행

`RunHarmony()` — PCA 공간에서 샘플별 분포를 반복적으로 보정합니다.

> **참고: Seurat v5의 두 가지 방법**
> - `RunHarmony()` ← 이 세미나 사용 (명확하고 빠름)
> - `IntegrateLayers(HarmonyIntegration)` — v5 네이티브, 결과 동일하지만 레이어 관리 필요

```
# SCTransform 방식 (더 정밀, 느림 — 고급)
# seurat_merged <- SCTransform(seurat_merged, vars.to.regress = c("S.Score","G2M.Score"))
# seurat_merged <- RunPCA(seurat_merged)
# seurat_merged <- RunHarmony(seurat_merged, group.by.vars = "sample")
```

In [ ]:
# Harmony 배치 보정
# group.by.vars: 보정할 배치 변수 (여기서는 샘플 ID)
seurat_merged <- RunHarmony(
  seurat_merged,
  group.by.vars    = "sample",   # 샘플별 배치 보정
  reduction        = "pca",      # PCA 결과를 입력으로 사용
  reduction.save   = "harmony",  # 결과 저장 위치
  max.iter.harmony = 20          # 최대 반복 횟수 (보통 10회 내에 수렴)
)

# 수렴 확인 — "Harmony converged after N iterations" 메시지 확인
cat("Harmony reduction dims:", ncol(Embeddings(seurat_merged, "harmony")), "\n")

**▶ 결과 해석**

Harmony 실행 후 확인할 내용:

- `converged after N iterations` — N이 작을수록 배치 효과가 뚜렷했음을 의미
- `max.iter` 도달 시 → `max.iter.harmony` 값을 늘려보세요
- 결과는 `seurat_merged@reductions$harmony` 에 저장됨 (PCA와 동일한 차원)

> **다음 단계**: 이 harmony embedding을 UMAP / Clustering 입력으로 사용합니다

## Step 9. UMAP

고차원 데이터를 **2D**로 시각화합니다.
Harmony 보정 결과(`reduction = "harmony"`)를 입력으로 사용합니다.

→ UMAP은 시각화 도구입니다. **클러스터링은 별도로 수행**하며 UMAP 좌표에 의존하지 않습니다.

In [ ]:
# UMAP (Harmony 통합 결과 기반)
seurat_merged <- RunUMAP(
  seurat_merged,
  reduction = "harmony",
  dims      = 1:30
)

# 통합 전/후 비교
p_before <- DimPlot(seurat_merged, reduction = "pca",  group.by = "sample") + ggtitle("Before Integration (PCA)")
p_after  <- DimPlot(seurat_merged, reduction = "umap", group.by = "sample") + ggtitle("After Integration (UMAP)")
p_before + p_after

**▶ 결과 해석**

| | Before Integration (PCA) | After Integration (UMAP) |
|--|--|--|
| 기대 패턴 | 샘플별로 분리됨 | 샘플이 섞여 하나의 구름 형성 |

- **Before**: Young/Old 샘플이 PCA에서 분리 → 배치 효과 확인
- **After**: Harmony 보정 후 같은 세포 타입끼리 모임 → 통합 성공

> UMAP에서 샘플이 여전히 분리된다면 → Harmony 파라미터 조정 필요

## Step 10. Clustering

**KNN 그래프** 기반 Louvain/Leiden 알고리즘으로 클러스터를 탐지합니다.
`resolution`이 **클수록 더 많은 클러스터**가 생성됩니다.

> **💡 Tip:** Resolution **0.4~0.6**에서 시작하세요. VlnPlot과 마커 유전자를 보고 최종 선택합니다.

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_16.png" width="850"/>

*Fig. 12 — Resolution에 따른 클러스터링 결과 비교 (Res 0.2 ~ 1.2)*

In [ ]:
# 이웃 그래프 생성 (Harmony 결과 기반)
seurat_merged <- FindNeighbors(
  seurat_merged,
  reduction = "harmony",
  dims      = 1:30
)

# 클러스터링 — resolution 숫자가 클수록 클러스터가 더 많이 나옵니다
seurat_merged <- FindClusters(seurat_merged, resolution = 0.2)
cat("Resolution 0.2 →", length(unique(seurat_merged$seurat_clusters)), "clusters\n")

seurat_merged <- FindClusters(seurat_merged, resolution = 0.4)
cat("Resolution 0.4 →", length(unique(seurat_merged$seurat_clusters)), "clusters\n")

seurat_merged <- FindClusters(seurat_merged, resolution = 0.6)
cat("Resolution 0.6 →", length(unique(seurat_merged$seurat_clusters)), "clusters\n")

seurat_merged <- FindClusters(seurat_merged, resolution = 0.8)
cat("Resolution 0.8 →", length(unique(seurat_merged$seurat_clusters)), "clusters\n")


In [ ]:
# 세미나에서는 Resolution 0.4 사용
Idents(seurat_merged) <- "RNA_snn_res.0.4"

DimPlot(seurat_merged, reduction = "umap", label = TRUE, label.size = 4) +
  ggtitle("클러스터 (Resolution 0.4)") +
  theme_minimal()


**▶ 결과 해석**

UMAP 위의 클러스터 번호를 확인하세요:

- **클러스터 수** — 생물학적으로 의미 있는 수? 너무 많거나 적지 않은지 확인
- **클러스터 크기** — 아주 작은 클러스터(< 50 cells)는 artifact일 수 있음
- **클러스터 모양** — 분리가 뚜렷하면 좋음; 퍼져있으면 resolution 조정 고려

> Resolution을 바꾸면서 UMAP을 비교해보세요 (0.2 vs 0.4 vs 0.8)

## Step 11. Marker Gene Identification

각 클러스터를 나머지와 비교하여 **대표 마커 유전자**를 찾습니다.

`FindAllMarkers()` 주요 파라미터:
- `only.pos = TRUE`: **해당 클러스터에서 높게 발현**되는 유전자만
- `min.pct = 0.25`: 최소 **25%** 세포에서 발현
- `logfc.threshold = 0.25`: 최소 **log2FC 0.25** 기준

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_17.png" width="850"/>

*Fig. 13 — 세포 타입 어노테이션 전략: 자동 어노테이션 vs 수동 어노테이션 (Clake ZA et al., Nat Protoc 2019)*

In [ ]:
# 마커 유전자 탐색 (시간 소요: 5~15분)
# only.pos = TRUE: 해당 클러스터에서 높게 발현되는 유전자만
markers <- FindAllMarkers(
  seurat_merged,
  only.pos          = TRUE,
  min.pct           = 0.25,  # 최소 25% 세포에서 발현
  logfc.threshold   = 0.25   # 최소 log2FC 0.25
)

# Top 5 마커 확인
markers %>%
  group_by(cluster) %>%
  slice_max(avg_log2FC, n = 5) %>%
  print(n = Inf)

**▶ 결과 해석**

`FindAllMarkers()` 주요 컬럼:

| 컬럼 | 의미 |
|------|------|
| `avg_log2FC` | 해당 클러스터 vs 나머지의 발현 배수 차이 (log2) |
| `pct.1` | 해당 클러스터에서 발현된 세포 비율 |
| `pct.2` | 나머지 클러스터에서 발현된 세포 비율 |
| `p_val_adj` | BH 보정 p-value |

- **좋은 마커**: 높은 avg_log2FC + 높은 pct.1 + 낮은 pct.2
- 각 클러스터 Top 1~3 유전자를 NCBI/GeneCards에서 검색해보세요

In [ ]:
# Top 10 마커 히트맵
top10 <- markers %>%
  group_by(cluster) %>%
  slice_max(avg_log2FC, n = 10)

DoHeatmap(seurat_merged, features = top10$gene) +
  theme(axis.text.y = element_text(size = 6))

**▶ 결과 해석**

각 클러스터의 Top 마커 유전자 발현 패턴을 확인하세요:

- **진한 색 블록**이 해당 클러스터에서만 나타나면 → 좋은 마커
- 여러 클러스터에 걸쳐 발현 → 범용 마커 (세포 타입 구별에 한계)

> 히트맵 패턴과 논문의 마커 유전자를 비교하여 세포 타입을 추론합니다
> (다음 Step 12 참고)

## Step 12. Cell Type Annotation

### 마커 유전자 — Voigt et al. 2022 (PMC10162434) 기반

| 세포 타입 | 마커 유전자 | 비고 |
|----------|------------|------|
| RGC | SNCG, ISL1, POU4F2 | BRN3 계열 + SNCG |
| Amacrine | TFAP2A, PAX6, GAD1 | — |
| Bipolar | CABP5, PRKCA, GRM6 | VSX2 제외 (Progenitor 혼용 위험) |
| Müller glia | GLUL, RLBP1, SLC1A3 | 3개 모두 고신뢰 마커 |
| Rod | RHO, NRL, RCVRN | NRL = rod master TF |
| Cone | OPN1LW, ARR3, GNGT2 | — |
| Horizontal | LHX1, ONECUT2, PROX1 | — |
| Progenitor | VSX2, FGF19, LIN28B | Young 샘플에 풍부 |
| Microglia | CX3CR1, P2RY12, TMEM119 | — |
| Endothelial | PECAM1, CDH5, VWF | — |

> **💡 Tip:** Bipolar에서 VSX2를 제거했습니다 — 발달기 Progenitor와 혼용 위험이 있어 CABP5/PRKCA로 구별합니다.

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_18.png" width="850"/>

*Fig. 14 — 세포 타입 어노테이션 방법 및 Tip 4: 연구자의 주관이 중요!*

### Task. 어노테이션 테스트 — 3개 세포 타입

논문에서 **마커 특이성 + 풍부도** 기준으로 선별:

| 세포 타입 | 마커 | 선별 이유 |
|----------|------|---------|
| **Müller glia** | GLUL, RLBP1, SLC1A3 | 3개 모두 독립적으로 Müller를 정의 |
| **Rod** | RHO, NRL, RCVRN | 성체 망막 최다 세포 + NRL은 rod master TF |
| **RGC** | SNCG, ISL1, POU4F2 | POU4F(BRN3) + SNCG 조합 = best marker set |

In [ ]:
# 논문 기반 마커 유전자 (Voigt et al. 2022, PMC10162434)
markers_Muller <- c("GLUL",  "RLBP1", "SLC1A3")   # Müller glia
markers_Rod    <- c("RHO",   "NRL",   "RCVRN")    # Rod photoreceptor
markers_RGC    <- c("SNCG",  "ISL1",  "POU4F2")   # Retinal Ganglion Cell

# UMAP 위에 유전자 발현 표시
FeaturePlot(seurat_merged, features = markers_Muller, ncol = 3) +
  plot_annotation(title = "Müller Glia 마커")


In [ ]:
# 클러스터 번호 → 세포 타입 이름 매핑
# ▶ FindAllMarkers 결과를 보고 여기를 수정하세요!
new_labels <- c(
  "0" = "Muller_glia",
  "1" = "Progenitor",
  "2" = "Bipolar",
  "3" = "RGC",
  "4" = "Amacrine",
  "5" = "Rod",
  "6" = "Cone",
  "7" = "Horizontal",
  "8" = "Microglia",
  "9" = "Unknown"
)

# 클러스터 번호를 세포 타입 이름으로 교체
seurat_merged$cell_type <- new_labels[as.character(seurat_merged$seurat_clusters)]

# 최종 UMAP
DimPlot(seurat_merged, reduction = "umap", group.by = "cell_type",
        label = TRUE, label.size = 3, repel = TRUE) +
  ggtitle("Cell Type Annotation") +
  theme_minimal()


In [ ]:
# Young vs Old 세포 타입 구성 비교
prop_table <- table(seurat_merged$group, seurat_merged$cell_type)
prop_df    <- as.data.frame(prop.table(prop_table, margin = 1))
colnames(prop_df) <- c("group", "cell_type", "proportion")

ggplot(prop_df, aes(x = group, y = proportion, fill = cell_type)) +
  geom_bar(stat = "identity") +
  labs(title = "Young vs Old: 세포 타입 구성",
       x = "그룹", y = "비율") +
  theme_minimal()


In [ ]:
# 최종 저장
saveRDS(seurat_merged, file = file.path(SAVE_DIR, "02_seurat_annotated.rds"))
cat("Saved: 02_seurat_annotated.rds\n")

# 요약
cat("\n=== Final Summary ===\n")
cat("Total cells:", ncol(seurat_merged), "\n")
cat("Cell types:\n")
print(table(seurat_merged$cell_type, seurat_merged$group))

---

## 세미나 완료

| 단계 | 완료 내용 |
|------|---------|
| Step 0 | 환경 설정 (Seurat, Harmony) |
| Step 1 | 10X 데이터 로드 (4개 샘플) |
| Step 2-3 | QC 시각화 + 필터링 |
| Step 4-5 | LogNormalize + HVG |
| Step 6 | Cell Cycle (optional) |
| Step 7 | Scaling + PCA |
| Step 8 | Harmony 통합 |
| Step 9 | UMAP |
| Step 10 | Clustering |
| Step 11 | FindAllMarkers |
| Step 12 | Cell Type Annotation |